# Data Cleaning for Fraud Detection

This notebook performs a production-aware cleaning pass on the credit card fraud dataset. In this project, data cleaning does not mean aggressively removing unusual behavior. Instead, it means improving data reliability while preserving rare but potentially meaningful fraud patterns. That distinction matters because fraud detection systems depend on minority-class signals that can be damaged by generic cleaning rules.

The goal of this stage is to confirm that the dataset is structurally trustworthy for downstream feature engineering, modeling, threshold tuning, and business decision logic such as `BLOCK`, `REVIEW`, and `APPROVE`. The notebook therefore emphasizes evidence-based cleaning, careful duplicate handling, explicit documentation of data quality checks, and a conservative approach to outliers and potential leakage.

## 1. Cleaning Objectives

This notebook focuses on the following objectives:

- load the fraud dataset using the project pipeline rather than ad hoc paths
- perform a full data quality audit before applying any cleaning action
- identify duplicates, missing values, hidden missing values, type inconsistencies, and low-information features
- remove exact duplicate rows while preserving the fraud class distribution
- document why some issues are fixed now and why others are intentionally deferred
- validate the cleaned dataset and save outputs for later stages of the pipeline

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Project root not found. Open the notebook from inside the project.")

project_root = find_project_root(Path.cwd())
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import (
    AMOUNT_COLUMN,
    CLEANED_DATA_FILE,
    NUMERICAL_COLUMNS,
    PCA_COLUMNS,
    PROJECT_ROOT,
    TARGET_COLUMN,
    TIME_COLUMN,
)
from src.data.data_cleaning import clean_creditcard_data, save_cleaned_data
from src.data.data_loader import load_raw_data
from src.data.data_validation import DataValidationError, validate_cleaned_data

NOTEBOOK_TABLES_DIR = PROJECT_ROOT / "reports" / "tables" / "04_data_cleaning"
NOTEBOOK_FIGURES_DIR = PROJECT_ROOT / "reports" / "figures" / "04_data_cleaning"
NOTEBOOK_TABLES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

LEGIT_COLOR = "#4C78A8"
FRAUD_COLOR = "#E45756"

print("Imports OK")
print(f"  Target column      : {TARGET_COLUMN}")
print(f"  Numerical columns  : {len(NUMERICAL_COLUMNS)}")
print(f"  Cleaned output     : {CLEANED_DATA_FILE}")
print(f"  Tables dir         : {NOTEBOOK_TABLES_DIR}")
print(f"  Figures dir        : {NOTEBOOK_FIGURES_DIR}")

## 2. Load Input Data

This cleaning stage starts from the project-managed raw dataset using the same loader used elsewhere in the pipeline. That keeps the workflow consistent with earlier validation work and avoids path-level inconsistencies that make notebooks harder to maintain.

In [ ]:
df = load_raw_data()

print("Input dataset shape:", df.shape)
df.head()

## 3. Initial Data Quality Audit

Before cleaning anything, the notebook performs a broad audit of structure, completeness, type quality, feature cardinality, summary statistics, and class balance. In fraud detection, this audit is important because overly aggressive cleaning can erase minority-class information or introduce subtle training bias.

In [ ]:
raw_row_count = len(df)
raw_column_count = df.shape[1]
raw_duplicate_count = int(df.duplicated().sum())
raw_missing_count = int(df.isna().sum().sum())
object_column_count = int((df.dtypes == "object").sum())
class_counts_before = df[TARGET_COLUMN].value_counts().sort_index()
class_percent_before = (df[TARGET_COLUMN].value_counts(normalize=True).sort_index() * 100).round(4)

initial_audit_summary = pd.DataFrame([
    {"metric": "row_count", "value": raw_row_count},
    {"metric": "column_count", "value": raw_column_count},
    {"metric": "duplicate_rows", "value": raw_duplicate_count},
    {"metric": "total_missing_values", "value": raw_missing_count},
    {"metric": "object_type_columns", "value": object_column_count},
    {"metric": "legit_count", "value": int(class_counts_before.get(0, 0))},
    {"metric": "fraud_count", "value": int(class_counts_before.get(1, 0))},
    {"metric": "fraud_rate_percent", "value": float(class_percent_before.get(1, 0.0))},
])

initial_audit_summary.to_csv(NOTEBOOK_TABLES_DIR / "initial_data_quality_audit.csv", index=False)
initial_audit_summary

The initial audit gives a compact view of whether the dataset is broadly usable before we inspect individual quality dimensions in detail.

## 4. Missing Value Assessment

Fraud pipelines need explicit missing-value review because missing or malformed values can break transformations, distort scaling, or silently affect downstream scoring behavior. This section checks standard nulls first and then checks for hidden missing values such as blank strings or placeholder tokens.

In [ ]:
missing_value_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(4),
}).sort_values(["missing_count", "missing_percent"], ascending=False)

missing_value_summary.to_csv(NOTEBOOK_TABLES_DIR / "missing_value_assessment.csv")
missing_value_summary.head(10)

In [ ]:
placeholder_tokens = {"", " ", "na", "n/a", "null", "none", "nan", "missing", "unknown", "?"}
object_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()

if object_columns:
    hidden_missing_counts = {
        column: int(
            df[column]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(placeholder_tokens)
            .sum()
        )
        for column in object_columns
    }
    hidden_missing_summary = pd.DataFrame.from_dict(hidden_missing_counts, orient="index", columns=["hidden_missing_count"])
else:
    hidden_missing_summary = pd.DataFrame(columns=["hidden_missing_count"])

hidden_missing_summary.to_csv(NOTEBOOK_TABLES_DIR / "hidden_missing_value_assessment.csv")
hidden_missing_summary

If missing-value checks return zeros across the dataset, that is a useful result rather than an empty exercise. It means imputation is not currently required, and we can state that conclusion with evidence instead of assuming cleanliness.

## 5. Data Types, Unique Values, and Descriptive Statistics

Type quality matters because a production fraud model expects predictable numeric inputs. This section checks the declared dtypes, numeric convertibility where expected, per-column uniqueness, and descriptive statistics to support later feature review.

In [ ]:
dtype_audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "is_expected_numeric": [column in NUMERICAL_COLUMNS or column == TARGET_COLUMN for column in df.columns],
    "nunique": df.nunique(dropna=False),
})

numeric_coercion_issues = {}
for column in [*NUMERICAL_COLUMNS, TARGET_COLUMN]:
    coerced = pd.to_numeric(df[column], errors="coerce")
    numeric_coercion_issues[column] = int(coerced.isna().sum() - df[column].isna().sum())

dtype_audit["numeric_coercion_issue_count"] = [numeric_coercion_issues.get(column, 0) for column in df.columns]
dtype_audit.to_csv(NOTEBOOK_TABLES_DIR / "dtype_audit.csv")
dtype_audit

In [ ]:
unique_value_summary = pd.DataFrame({
    "nunique": df.nunique(dropna=False),
    "unique_ratio_percent": ((df.nunique(dropna=False) / len(df)) * 100).round(4),
}).sort_values("nunique")

unique_value_summary.to_csv(NOTEBOOK_TABLES_DIR / "unique_value_summary.csv")
unique_value_summary.head(10)

In [ ]:
descriptive_statistics = df.describe().T
descriptive_statistics.to_csv(NOTEBOOK_TABLES_DIR / "descriptive_statistics.csv")
descriptive_statistics.head()

## 6. Class Balance Before Cleaning

Before changing the data, we record the class distribution. Fraud detection datasets are intentionally imbalanced, and that imbalance is a modeling challenge to manage carefully later rather than a data-quality error to clean away.

In [ ]:
class_distribution_before = pd.DataFrame({
    "count": class_counts_before,
    "percent": class_percent_before,
})
class_distribution_before.index.name = TARGET_COLUMN
class_distribution_before.to_csv(NOTEBOOK_TABLES_DIR / "class_distribution_before_cleaning.csv")
class_distribution_before

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
class_labels = ["Legit (0)", "Fraud (1)"]
class_colors = [LEGIT_COLOR, FRAUD_COLOR]

axes[0].bar(class_labels, class_counts_before.values, color=class_colors)
axes[0].set_title("Class Distribution Before Cleaning")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Transaction Count")
for i, value in enumerate(class_counts_before.values):
    axes[0].text(i, value, f"{value}", ha="center", va="bottom")

axes[1].pie(
    class_counts_before.values,
    labels=[f"{label}\n{pct:.4f}%" for label, pct in zip(class_labels, class_percent_before.values)],
    colors=class_colors,
    startangle=90,
    counterclock=False,
)
axes[1].set_title("Class Share Before Cleaning")

plt.tight_layout()
plt.savefig(NOTEBOOK_FIGURES_DIR / "class_distribution_before_cleaning.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Duplicate Analysis and Cleaning Decision

Exact duplicates are especially dangerous in fraud detection because they can distort class frequencies, create overly optimistic validation results, and leak repeated records across train-test splits. This section therefore checks the impact of duplicates before removal and documents how much class composition changes when exact duplicates are removed.

In [ ]:
duplicate_mask = df.duplicated(keep="first")
duplicate_rows = df[duplicate_mask].copy()

duplicate_class_counts = duplicate_rows[TARGET_COLUMN].value_counts().sort_index() if not duplicate_rows.empty else pd.Series(dtype="int64")
duplicate_class_percent = (duplicate_rows[TARGET_COLUMN].value_counts(normalize=True).sort_index() * 100).round(4) if not duplicate_rows.empty else pd.Series(dtype="float64")

duplicate_impact_summary = pd.DataFrame({
    "duplicate_count": duplicate_class_counts,
    "duplicate_percent": duplicate_class_percent,
})
duplicate_impact_summary.index.name = TARGET_COLUMN
duplicate_impact_summary.to_csv(NOTEBOOK_TABLES_DIR / "duplicate_impact_summary.csv")
duplicate_impact_summary

The cleaning action in this notebook is intentionally narrow: remove exact duplicate rows, preserve the first occurrence, and leave the remaining feature values unchanged. This is a defensible early-stage cleaning step because it improves data integrity without forcing subjective edits onto potentially informative transactions.

In [ ]:
df_cleaned = clean_creditcard_data(df)

cleaned_row_count = len(df_cleaned)
rows_removed = raw_row_count - cleaned_row_count
remaining_duplicates = int(df_cleaned.duplicated().sum())

class_counts_after = df_cleaned[TARGET_COLUMN].value_counts().sort_index()
class_percent_after = (df_cleaned[TARGET_COLUMN].value_counts(normalize=True).sort_index() * 100).round(4)

duplicate_removal_summary = pd.DataFrame([
    {"metric": "rows_before_cleaning", "value": raw_row_count},
    {"metric": "rows_after_cleaning", "value": cleaned_row_count},
    {"metric": "rows_removed_as_exact_duplicates", "value": rows_removed},
    {"metric": "remaining_duplicates", "value": remaining_duplicates},
    {"metric": "fraud_rate_before_percent", "value": float(class_percent_before.get(1, 0.0))},
    {"metric": "fraud_rate_after_percent", "value": float(class_percent_after.get(1, 0.0))},
])

duplicate_removal_summary.to_csv(NOTEBOOK_TABLES_DIR / "duplicate_removal_summary.csv", index=False)
duplicate_removal_summary

In [ ]:
class_distribution_after = pd.DataFrame({
    "count": class_counts_after,
    "percent": class_percent_after,
})
class_distribution_after.index.name = TARGET_COLUMN
class_distribution_after.to_csv(NOTEBOOK_TABLES_DIR / "class_distribution_after_cleaning.csv")

class_distribution_change_summary = pd.DataFrame({
    "count_before": class_counts_before,
    "count_after": class_counts_after,
    "percent_before": class_percent_before,
    "percent_after": class_percent_after,
    "count_change": class_counts_after - class_counts_before,
    "percent_point_change": (class_percent_after - class_percent_before).round(6),
})
class_distribution_change_summary.index.name = TARGET_COLUMN
class_distribution_change_summary.to_csv(NOTEBOOK_TABLES_DIR / "class_distribution_change_summary.csv")
class_distribution_change_summary

In [ ]:
plot_df = pd.DataFrame({
    "Before Cleaning": class_counts_before,
    "After Cleaning": class_counts_after,
})
plot_df.index = ["Legit (0)", "Fraud (1)"]

ax = plot_df.plot(kind="bar", figsize=(8, 5), color=[LEGIT_COLOR, FRAUD_COLOR])
ax.set_title("Class Distribution Before vs After Duplicate Removal")
ax.set_xlabel("Class")
ax.set_ylabel("Transaction Count")
ax.set_xticklabels(plot_df.index, rotation=0)
plt.tight_layout()
plt.savefig(NOTEBOOK_FIGURES_DIR / "class_distribution_before_after_cleaning.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Data Type Validation After Cleaning

Type validation is repeated after cleaning because pipeline reliability depends on consistent numeric inputs. If this stage leaves unexpected object columns, malformed numeric values, or invalid target labels, those issues should be surfaced before feature engineering begins.

In [ ]:
cleaned_dtype_audit = pd.DataFrame({
    "dtype": df_cleaned.dtypes.astype(str),
    "nunique": df_cleaned.nunique(dropna=False),
})
cleaned_dtype_audit["unexpected_object_dtype"] = [dtype == "object" for dtype in cleaned_dtype_audit["dtype"]]
cleaned_dtype_audit["numeric_coercion_issue_count"] = [
    int(pd.to_numeric(df_cleaned[column], errors="coerce").isna().sum() - df_cleaned[column].isna().sum()) if column in [*NUMERICAL_COLUMNS, TARGET_COLUMN] else 0
    for column in df_cleaned.columns
]
cleaned_dtype_audit.to_csv(NOTEBOOK_TABLES_DIR / "cleaned_dtype_audit.csv")
cleaned_dtype_audit

If these checks show no numeric coercion issues and no unexpected object columns, we can explicitly say the dataset remains type-consistent for the next stage.

## 9. Feature Sanity Checks

Feature sanity checks look for impossible values, suspicious ranges, and low-information features. The goal is not to aggressively prune features in the cleaning notebook, but to document anything that could affect later modeling decisions.

In [ ]:
feature_range_summary = pd.DataFrame({
    "column": [TIME_COLUMN, *PCA_COLUMNS, AMOUNT_COLUMN],
    "min": [df_cleaned[column].min() for column in [TIME_COLUMN, *PCA_COLUMNS, AMOUNT_COLUMN]],
    "max": [df_cleaned[column].max() for column in [TIME_COLUMN, *PCA_COLUMNS, AMOUNT_COLUMN]],
    "mean": [df_cleaned[column].mean() for column in [TIME_COLUMN, *PCA_COLUMNS, AMOUNT_COLUMN]],
})
feature_range_summary["abs_max"] = feature_range_summary[["min", "max"]].abs().max(axis=1)
feature_range_summary["suspicious_abs_max_gt_100"] = feature_range_summary["abs_max"] > 100
feature_range_summary.to_csv(NOTEBOOK_TABLES_DIR / "feature_range_summary.csv", index=False)
feature_range_summary.head()

In [ ]:
impossible_value_checks = pd.DataFrame([
    {"check": "time_negative_count", "value": int((pd.to_numeric(df_cleaned[TIME_COLUMN], errors="coerce") < 0).sum())},
    {"check": "amount_negative_count", "value": int((pd.to_numeric(df_cleaned[AMOUNT_COLUMN], errors="coerce") < 0).sum())},
    {"check": "invalid_target_count", "value": int((~df_cleaned[TARGET_COLUMN].isin([0, 1])).sum())},
])
impossible_value_checks.to_csv(NOTEBOOK_TABLES_DIR / "impossible_value_checks.csv", index=False)
impossible_value_checks

In [ ]:
constant_feature_summary = pd.DataFrame({
    "column": df_cleaned.columns,
    "nunique": df_cleaned.nunique(dropna=False).values,
})
constant_feature_summary["is_constant"] = constant_feature_summary["nunique"] == 1
constant_feature_summary.to_csv(NOTEBOOK_TABLES_DIR / "constant_feature_summary.csv", index=False)
constant_feature_summary[constant_feature_summary["is_constant"]]

In [ ]:
near_constant_rows = []
for column in df_cleaned.columns:
    top_frequency_percent = float(df_cleaned[column].value_counts(normalize=True, dropna=False).iloc[0] * 100)
    variance_value = float(df_cleaned[column].var()) if column in [*NUMERICAL_COLUMNS, TARGET_COLUMN] else np.nan
    near_constant_rows.append({
        "column": column,
        "top_frequency_percent": round(top_frequency_percent, 4),
        "variance": variance_value,
        "is_near_constant_top_frequency": top_frequency_percent >= 99.9,
        "is_low_variance_numeric": variance_value < 1e-6 if not np.isnan(variance_value) else False,
    })

near_constant_feature_summary = pd.DataFrame(near_constant_rows)
near_constant_feature_summary.to_csv(NOTEBOOK_TABLES_DIR / "near_constant_feature_summary.csv", index=False)
near_constant_feature_summary[
    near_constant_feature_summary["is_near_constant_top_frequency"] | near_constant_feature_summary["is_low_variance_numeric"]
]

Features flagged as constant or near-constant are documented here, but not automatically removed. In a fraud project, even low-variance variables should be reviewed in modeling context before exclusion because business value and interaction effects can matter more than standalone variance.

## 10. Outlier Awareness

Outliers are not removed blindly in this notebook. In fraud detection, rare and extreme behavior may be the signal we want the model to learn. This section therefore documents distribution tails and range behavior, but intentionally defers any aggressive outlier treatment until later stages where business context, model behavior, and evaluation trade-offs can be considered together.

In [ ]:
outlier_awareness_summary = pd.DataFrame({
    "metric": [
        "amount_p50", "amount_p75", "amount_p90", "amount_p95", "amount_p99", "amount_p999", "amount_max",
        "time_p50", "time_p75", "time_p90", "time_p95", "time_p99", "time_p999", "time_max"
    ],
    "value": [
        df_cleaned[AMOUNT_COLUMN].quantile(0.50),
        df_cleaned[AMOUNT_COLUMN].quantile(0.75),
        df_cleaned[AMOUNT_COLUMN].quantile(0.90),
        df_cleaned[AMOUNT_COLUMN].quantile(0.95),
        df_cleaned[AMOUNT_COLUMN].quantile(0.99),
        df_cleaned[AMOUNT_COLUMN].quantile(0.999),
        df_cleaned[AMOUNT_COLUMN].max(),
        df_cleaned[TIME_COLUMN].quantile(0.50),
        df_cleaned[TIME_COLUMN].quantile(0.75),
        df_cleaned[TIME_COLUMN].quantile(0.90),
        df_cleaned[TIME_COLUMN].quantile(0.95),
        df_cleaned[TIME_COLUMN].quantile(0.99),
        df_cleaned[TIME_COLUMN].quantile(0.999),
        df_cleaned[TIME_COLUMN].max(),
    ],
})

outlier_awareness_summary.to_csv(NOTEBOOK_TABLES_DIR / "outlier_awareness_summary.csv", index=False)
outlier_awareness_summary

Professional note: extreme values in `Amount`, `Time`, or PCA-derived dimensions may reflect unusual but legitimate fraud signatures. For that reason, this notebook treats outliers as review items rather than automatic deletion candidates.

## 11. Leakage Awareness

Leakage checks are documented even when no obvious issue is found. For a deployable fraud system, it is not enough to avoid leakage accidentally; the review should show that no feature appears to directly encode the target or reveal information that would not be available at scoring time.

In [ ]:
target_correlations = (
    df_cleaned[[*NUMERICAL_COLUMNS, TARGET_COLUMN]]
    .corr(numeric_only=True)[TARGET_COLUMN]
    .drop(TARGET_COLUMN)
    .abs()
    .sort_values(ascending=False)
)

leakage_review_summary = pd.DataFrame({
    "feature": target_correlations.index,
    "abs_correlation_with_target": target_correlations.values,
})
leakage_review_summary["review_flag_gt_0_95"] = leakage_review_summary["abs_correlation_with_target"] > 0.95
leakage_review_summary.to_csv(NOTEBOOK_TABLES_DIR / "leakage_review_summary.csv", index=False)
leakage_review_summary.head(10)

A high correlation alone does not prove leakage, and a low correlation does not guarantee safety. However, this review gives a documented first pass. In this dataset, no obvious post-event, approval-outcome, or target-encoded fields are expected from the schema, so leakage risk appears low at this stage unless later feature engineering introduces it.

## 12. Cleaning Actions Summary

A good cleaning notebook should say not only what changed, but also what was intentionally left unchanged and why. That is especially important in fraud detection, where over-cleaning can reduce the very signals the model needs.

In [ ]:
cleaning_actions_summary = pd.DataFrame([
    {
        "area": "Missing values",
        "status": "Checked",
        "action_taken": "No imputation applied if missing-value counts remain zero.",
        "reason": "No evidence-based need to alter the data when completeness is already acceptable."
    },
    {
        "area": "Hidden missing values",
        "status": "Checked",
        "action_taken": "Placeholder-token scan performed for object columns.",
        "reason": "Malformed placeholders can break production preprocessing even when null counts are zero."
    },
    {
        "area": "Duplicates",
        "status": "Cleaned",
        "action_taken": "Exact duplicate rows removed.",
        "reason": "Duplicates can bias model training, evaluation, and fraud-rate estimates."
    },
    {
        "area": "Outliers",
        "status": "Deferred",
        "action_taken": "No blanket outlier removal applied.",
        "reason": "Extreme transactions may be meaningful fraud signals and should be reviewed with modeling context."
    },
    {
        "area": "Low-information features",
        "status": "Reviewed",
        "action_taken": "Constant and near-constant features documented rather than dropped automatically.",
        "reason": "Final feature pruning should be tied to downstream modeling evidence."
    },
    {
        "area": "Leakage risk",
        "status": "Reviewed",
        "action_taken": "Schema-level and correlation-based review documented.",
        "reason": "Leakage must be controlled before training and deployment, not discovered after strong validation scores."
    },
])

cleaning_actions_summary.to_csv(NOTEBOOK_TABLES_DIR / "cleaning_actions_summary.csv", index=False)
cleaning_actions_summary

## 13. Final Validation After Cleaning

The final validation stage confirms that the cleaned dataset is internally consistent and ready for the next pipeline step. This includes verifying final shape, duplicate removal, remaining missing values, class distribution, data types, and the formal cleaned-data validator.

In [ ]:
final_missing_count = int(df_cleaned.isna().sum().sum())
final_object_column_count = int((df_cleaned.dtypes == "object").sum())

try:
    validate_cleaned_data(df_cleaned)
    final_validation_status = "PASS"
    final_validation_message = "Cleaned dataset passed the implemented validation checks."
except DataValidationError as error:
    final_validation_status = "FAIL"
    final_validation_message = str(error)

final_validation_summary = pd.DataFrame([
    {"metric": "final_row_count", "value": len(df_cleaned)},
    {"metric": "final_column_count", "value": df_cleaned.shape[1]},
    {"metric": "remaining_duplicates", "value": remaining_duplicates},
    {"metric": "final_missing_values", "value": final_missing_count},
    {"metric": "final_object_columns", "value": final_object_column_count},
    {"metric": "fraud_rate_after_cleaning_percent", "value": float(class_percent_after.get(1, 0.0))},
    {"metric": "final_validation_status", "value": final_validation_status},
    {"metric": "ready_for_feature_engineering", "value": "YES" if final_validation_status == "PASS" else "NO"},
])

final_validation_summary.to_csv(NOTEBOOK_TABLES_DIR / "final_validation_summary.csv", index=False)
final_validation_summary

## 14. Save Cleaned Outputs

Outputs are saved using the project utilities and report directories so the cleaning stage remains reproducible and consistent with the rest of the pipeline.

In [ ]:
cleaned_data_path = save_cleaned_data(df_cleaned)
print(f"Cleaned dataset saved to: {cleaned_data_path}")

## 15. Final Conclusion

This cleaning stage improves data reliability by removing exact duplicate transactions, documenting completeness and type quality, and checking for feature-level risks without over-cleaning the fraud signal. The minority fraud class is preserved rather than normalized away, which keeps the dataset suitable for later feature engineering, imbalance-aware modeling, threshold tuning, and business decision rules such as `BLOCK`, `REVIEW`, and `APPROVE`. From a production perspective, the dataset is now better positioned for a deployable API pipeline because the cleaning decisions are explicit, reproducible, and aligned with real scoring-time data quality requirements.

## 16. Save Cleaning Report

In [ ]:
cleaning_report = f"""# Cleaning Report

## Project Context

This cleaning stage prepares the credit card fraud dataset for downstream feature engineering, modeling, threshold tuning, and decision logic design. The approach is intentionally conservative because fraud datasets contain rare but valuable minority-class patterns that can be damaged by over-cleaning.

## Initial Audit Summary

- Input shape: {df.shape}
- Duplicate rows before cleaning: {raw_duplicate_count}
- Total missing values before cleaning: {raw_missing_count}
- Object-type columns before cleaning: {object_column_count}
- Fraud rate before cleaning: {float(class_percent_before.get(1, 0.0)):.4f}%

## Cleaning Decisions

- Exact duplicate rows were removed.
- No blanket missing-value imputation was applied if the dataset remained complete.
- No blanket outlier removal was applied because rare extreme patterns may be meaningful fraud signals.
- Low-information features and leakage risk were reviewed and documented rather than changed automatically.

## Final Validation Summary

- Output shape: {df_cleaned.shape}
- Remaining duplicates: {remaining_duplicates}
- Total missing values after cleaning: {final_missing_count}
- Final validation status: {final_validation_status}
- Final validation message: {final_validation_message}
- Fraud rate after cleaning: {float(class_percent_after.get(1, 0.0)):.4f}%

## Modeling Readiness

The cleaned dataset is more reliable for fraud modeling because duplicate leakage risk has been reduced while minority-class structure has been preserved. The dataset is ready for feature engineering and later modeling work, where imbalance-aware evaluation, threshold tuning, and business decision mapping can be handled more directly.

## Saved Tables

- `reports/tables/04_data_cleaning/initial_data_quality_audit.csv`
- `reports/tables/04_data_cleaning/missing_value_assessment.csv`
- `reports/tables/04_data_cleaning/hidden_missing_value_assessment.csv`
- `reports/tables/04_data_cleaning/dtype_audit.csv`
- `reports/tables/04_data_cleaning/unique_value_summary.csv`
- `reports/tables/04_data_cleaning/descriptive_statistics.csv`
- `reports/tables/04_data_cleaning/class_distribution_before_cleaning.csv`
- `reports/tables/04_data_cleaning/duplicate_impact_summary.csv`
- `reports/tables/04_data_cleaning/duplicate_removal_summary.csv`
- `reports/tables/04_data_cleaning/class_distribution_after_cleaning.csv`
- `reports/tables/04_data_cleaning/class_distribution_change_summary.csv`
- `reports/tables/04_data_cleaning/cleaned_dtype_audit.csv`
- `reports/tables/04_data_cleaning/feature_range_summary.csv`
- `reports/tables/04_data_cleaning/impossible_value_checks.csv`
- `reports/tables/04_data_cleaning/constant_feature_summary.csv`
- `reports/tables/04_data_cleaning/near_constant_feature_summary.csv`
- `reports/tables/04_data_cleaning/outlier_awareness_summary.csv`
- `reports/tables/04_data_cleaning/leakage_review_summary.csv`
- `reports/tables/04_data_cleaning/cleaning_actions_summary.csv`
- `reports/tables/04_data_cleaning/final_validation_summary.csv`
- `reports/tables/04_data_cleaning/cleaning_report.md`

## Saved Figures

- `reports/figures/04_data_cleaning/class_distribution_before_cleaning.png`
- `reports/figures/04_data_cleaning/class_distribution_before_after_cleaning.png`

## Final Judgment

The cleaned dataset is structurally stronger and more reliable for fraud modeling than the raw input because exact duplicates have been removed and data quality checks have been documented comprehensively. The fraud class distribution has been preserved, aggressive outlier removal has been avoided, and the dataset is ready for feature engineering, modeling, threshold tuning, and future deployment-oriented scoring workflows.
"""

report_path = NOTEBOOK_TABLES_DIR / "cleaning_report.md"
report_path.write_text(cleaning_report, encoding="utf-8")
print(f"Cleaning report saved to: {report_path}")

## 17. Recommendations

Recommended next actions for the pipeline:

- start `05_feature_engineering.ipynb` from the cleaned dataset rather than the raw file
- preserve the class imbalance during preprocessing and handle it with evaluation strategy, resampling choices, or class weighting later
- review low-information features again in the context of model behavior before dropping them
- treat any future outlier handling as a modeling decision, not a generic cleaning step
- re-check leakage whenever new engineered variables are introduced